# Deep learning project

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, ConcatDataset
import segmentation_models_pytorch as smp

from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import os
from PIL import Image
from road_dataset import RoadImageDataset, train_transform, val_transform

root_massachusetts = r"C:\Dev\deeplearning\data\massachusetts-roads-dataset"
root_deepglobe = r"C:\Dev\deeplearning\data\deepglobe"

# Windows requires this block for multiprocessing
if __name__ == "__main__":
    print("[+] loading datasets...")

    SEED = 42
    mass_train = RoadImageDataset(root_massachusetts, "train", train_transform)
    mass_val = RoadImageDataset(root_massachusetts, "val", val_transform)

    dg_train = RoadImageDataset(
        root_deepglobe, "train", train_transform, val_fraction=0.1, seed=SEED
    )
    dg_val = RoadImageDataset(
        root_deepglobe, "val", val_transform, val_fraction=0.1, seed=SEED
    )

    train_dataset = ConcatDataset([mass_train, dg_train])
    print(
        f"[+] train: {len(mass_train)} mass + {len(dg_train)} dg = {len(train_dataset)}"
    )

    val_dataset = ConcatDataset([mass_val, dg_val])

    # Define per-sample weights
    weight_mass = 1.0 / len(mass_train)
    weight_dg = 1.0 / len(dg_train)

    # Build the combined weight list
    weights = [weight_mass] * len(mass_train) + [weight_dg] * len(dg_train)

    # Create the sampler (replacement=True is mandatory for WeightedRandomSampler)
    sampler = WeightedRandomSampler(
        weights=weights, num_samples=len(train_dataset), replacement=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        sampler=sampler,  # Use sampler here instead of shuffle=True
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
        drop_last=True,
    )

    val_loaders = {
        "mass": DataLoader(
            mass_val,
            batch_size=8,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
        ),
        "dg": DataLoader(
            dg_val,
            batch_size=8,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
        ),
    }

    # Model definition
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    )

    # Pre-training initialization
    print("[+] started initializations...")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")

    device = torch.device("cuda")
    model = model.to(device, memory_format=torch.channels_last)

    criterion = smp.losses.DiceLoss(mode="binary", from_logits=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, fused=True)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    print("[+] starting the training...")
    num_epochs = 10
    best_val = float("inf")
    ckpt_dir = r"C:\Dev\deeplearning\trained_models"

    print("[+] starting epoch...")
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        print("starting batching...")
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images = images.to(
                device, non_blocking=True, memory_format=torch.channels_last
            )
            labels = labels.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(images)

            # Loss computed on fp32 logits
            loss = criterion(outputs.float(), labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}")

        model.eval()
        val_losses = {}

        with torch.no_grad():
            for name, loader in val_loaders.items():
                total = torch.tensor(0.0, device=device)
                inter = torch.tensor(0, device=device)
                union = torch.tensor(0, device=device)

                for images, labels in loader:
                    images = images.to(
                        device,
                        non_blocking=True,
                        memory_format=torch.channels_last,
                    )
                    labels = labels.to(device, non_blocking=True).float()

                    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                        outputs = model(images)

                    outputs = outputs.float()
                    total += criterion(outputs, labels)

                    pred = torch.sigmoid(outputs) > 0.5
                    tgt = labels.bool()
                    inter += (pred & tgt).sum()
                    union += (pred | tgt).sum()

                val_losses[name] = (total / len(loader)).item()
                iou = (inter / union).item() if union > 0 else 0.0
                print(f"  {name}: loss {val_losses[name]:.4f} | IoU {iou:.4f}")

        avg_val_loss = sum(val_losses.values()) / len(val_losses)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val:
            best_val = avg_val_loss
            os.makedirs(ckpt_dir, exist_ok=True)
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_loss": avg_train_loss,
                    "val_loss": avg_val_loss,
                    "val_losses": val_losses,
                },
                os.path.join(ckpt_dir, "roadseg_merged_dice_best.pt"),
            )
            print(f"  [*] new best ({best_val:.4f}) — saved")